## Objectif du notebook

Ce notebook prépare les fichiers nécessaires à la construction du tableau de bord Power BI du volet B Data Analyst.

Les fichiers générés sont des tables agrégées, plus légères que le fichier complet `consommation_preparee.csv`.

Les exports sont destinés à alimenter plusieurs vues décisionnelles :
- vue direction ;
- vue exploitation réseau ;
- vue financière ;
- vue relation client ;
- vue qualité des données.

Les fichiers produits seront stockés dans `volet-b-data-analyst/data_cleaned/`.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

In [2]:
ROOT_DIR = Path.cwd()

if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parents[1]

DATA_CLEANED_DIR = ROOT_DIR / "volet-b-data-analyst" / "data_cleaned"

DATA_CLEANED_DIR

WindowsPath('c:/Users/Master/Desktop/Mohamed/1CPDA/Examen_S2/ExaS2/neovolt-grid-plus/volet-b-data-analyst/data_cleaned')

In [3]:
# Chargement des fichiers préparés

conso = pd.read_csv(DATA_CLEANED_DIR / "consommation_preparee.csv")
incidents = pd.read_csv(DATA_CLEANED_DIR / "incidents_preparees.csv")
releves_horaires = pd.read_csv(DATA_CLEANED_DIR / "releves_horaires_preparees.csv")
reclamations = pd.read_csv(DATA_CLEANED_DIR / "reclamations_preparees.csv")
fraudes = pd.read_csv(DATA_CLEANED_DIR / "cas_fraude_confirmes_preparees.csv")
rapport_nettoyage = pd.read_csv(DATA_CLEANED_DIR / "rapport_nettoyage.csv")

print("conso :", conso.shape)
print("incidents :", incidents.shape)
print("releves_horaires :", releves_horaires.shape)
print("reclamations :", reclamations.shape)
print("fraudes :", fraudes.shape)
print("rapport_nettoyage :", rapport_nettoyage.shape)

conso : (511700, 36)
incidents : (420, 10)
releves_horaires : (21600, 8)
reclamations : (3000, 9)
fraudes : (24, 4)
rapport_nettoyage : (7, 2)


In [4]:
# Conversion des dates

conso["date"] = pd.to_datetime(conso["date"], errors="coerce")
incidents["date"] = pd.to_datetime(incidents["date"], errors="coerce")
incidents["date_debut"] = pd.to_datetime(incidents["date_debut"], errors="coerce")
releves_horaires["date"] = pd.to_datetime(releves_horaires["date"], errors="coerce")
releves_horaires["horodatage"] = pd.to_datetime(releves_horaires["horodatage"], errors="coerce")
reclamations["date"] = pd.to_datetime(reclamations["date"], errors="coerce")
fraudes["date_detection"] = pd.to_datetime(fraudes["date_detection"], errors="coerce")

Le fichier complet `consommation_preparee.csv` est volumineux et n’est pas adapté à un dépôt GitHub.

Pour Power BI, la stratégie retenue est de produire des tables agrégées :
- consommation mensuelle ;
- consommation par zone ;
- consommation par type de client ;
- consommation par segment ;
- consommation par saison ;
- météo et consommation journalière ;
- incidents par zone ;
- réclamations mensuelles ;
- profil horaire moyen ;
- indicateurs de qualité.

In [5]:
# Table KPI globale pour Power BI

kpi_global = pd.DataFrame([
    {
        "indicateur": "Nombre de relevés",
        "valeur": len(conso)
    },
    {
        "indicateur": "Nombre de PDL",
        "valeur": conso["id_pdl"].nunique()
    },
    {
        "indicateur": "Nombre de clients",
        "valeur": conso["id_client"].nunique()
    },
    {
        "indicateur": "Consommation totale nettoyée kWh",
        "valeur": round(conso["consommation_kwh_clean"].sum(), 2)
    },
    {
        "indicateur": "Consommation moyenne journalière kWh",
        "valeur": round(conso["consommation_kwh_clean"].mean(), 2)
    },
    {
        "indicateur": "Part de lignes imputées %",
        "valeur": round(conso["flag_conso_imputee"].mean() * 100, 2)
    },
    {
        "indicateur": "Nombre d'incidents réseau",
        "valeur": len(incidents)
    },
    {
        "indicateur": "Nombre de réclamations",
        "valeur": len(reclamations)
    },
    {
        "indicateur": "Satisfaction moyenne",
        "valeur": round(reclamations["satisfaction"].mean(), 2)
    }
])

kpi_global

,indicateur,valeur
0,Nombre de relevés,511700.00
1,Nombre de PDL,700.00
2,Nombre de clients,700.00
3,Consommation totale nettoyée kWh,12186856.03
4,Consommation moyenne journalière kWh,23.82
5,Part de lignes imputées %,12.13
6,Nombre d'incidents réseau,420.00
7,Nombre de réclamations,3000.00
8,Satisfaction moyenne,2.45


In [6]:
# Consommation mensuelle

consommation_mensuelle = (
    conso
    .groupby(["annee", "mois"], as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        nb_releves=("consommation_kwh_clean", "count"),
        nb_pdl=("id_pdl", "nunique"),
        temp_moyenne_c=("temp_moyenne_c", "mean"),
        degres_jour_chauffage=("degres_jour_chauffage", "mean")
    )
)

consommation_mensuelle["periode"] = pd.to_datetime(
    consommation_mensuelle["annee"].astype(str) + "-"
    + consommation_mensuelle["mois"].astype(str) + "-01"
)

consommation_mensuelle.head()

,annee,mois,consommation_totale_kwh,consommation_moyenne_kwh,nb_releves,nb_pdl,temp_moyenne_c,degres_jour_chauffage,periode
0,2024,1,626755.020,28.882720,21700,700,3.425959,13.574041,2024-01-01
1,2024,2,573341.275,28.243413,20300,700,4.403022,12.596978,2024-02-01
2,2024,3,566391.855,26.101007,21700,700,7.306348,9.693652,2024-03-01
3,2024,4,500094.545,23.814026,21000,700,11.857221,5.157233,2024-04-01
4,2024,5,460200.220,21.207383,21700,700,16.562511,1.297673,2024-05-01


In [7]:
# Consommation par zone

consommation_par_zone = (
    conso
    .groupby("zone", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        consommation_mediane_kwh=("consommation_kwh_clean", "median"),
        nb_pdl=("id_pdl", "nunique"),
        nb_clients=("id_client", "nunique"),
        part_lignes_imputees=("flag_conso_imputee", "mean")
    )
)

consommation_par_zone["part_lignes_imputees_pct"] = round(
    consommation_par_zone["part_lignes_imputees"] * 100, 2
)

consommation_par_zone = consommation_par_zone.drop(columns=["part_lignes_imputees"])

consommation_par_zone.head()

,zone,consommation_totale_kwh,consommation_moyenne_kwh,consommation_mediane_kwh,nb_pdl,nb_clients,part_lignes_imputees_pct
0,Bourg-Ancien,1383393.480,24.262399,12.99,78,78,11.44
1,Centre-Ville,1596861.085,21.628599,11.56,101,101,12.50
2,Coteaux-Ouest,984087.950,16.219537,9.85,83,83,2.67
3,Parc-Tertiaire,2924597.025,40.412290,38.55,99,99,13.22
4,Plateau-Est,776579.595,13.115462,9.64,81,81,2.74


In [8]:
# Consommation par type de client

consommation_par_type_client = (
    conso
    .groupby("type_client", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        consommation_mediane_kwh=("consommation_kwh_clean", "median"),
        nb_pdl=("id_pdl", "nunique"),
        nb_clients=("id_client", "nunique")
    )
)

# Consommation par segment

consommation_par_segment = (
    conso
    .groupby("segment", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        consommation_mediane_kwh=("consommation_kwh_clean", "median"),
        nb_clients=("id_client", "nunique"),
        nb_pdl=("id_pdl", "nunique")
    )
)

# Consommation par type de chauffage

consommation_par_chauffage = (
    conso
    .groupby("type_chauffage", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        consommation_mediane_kwh=("consommation_kwh_clean", "median"),
        nb_pdl=("id_pdl", "nunique")
    )
)

consommation_par_type_client.head(), consommation_par_segment.head(), consommation_par_chauffage.head()

(     type_client  consommation_totale_kwh  consommation_moyenne_kwh  \
 0     industriel              1464109.000                 31.791827   
 1  professionnel              7345927.150                 47.626294   
 2    residentiel              3376819.885                 10.843786   
 
    consommation_mediane_kwh  nb_pdl  nb_clients  
 0                     34.75      63          63  
 1                     46.31     211         211  
 2                      9.32     426         426  ,
         segment  consommation_totale_kwh  consommation_moyenne_kwh  \
 0  collectivite               573550.450                 35.664124   
 1    entreprise              3640174.050                 41.154696   
 2   particulier              3376819.885                 10.843786   
 3     petit_pro              4596311.650                 47.997741   
 
    consommation_mediane_kwh  nb_clients  nb_pdl  
 0                    36.365          22      22  
 1                    39.110         121     1

In [9]:
# Consommation par saison

consommation_par_saison = (
    conso
    .groupby("saison", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        temp_moyenne_c=("temp_moyenne_c", "mean"),
        degres_jour_chauffage=("degres_jour_chauffage", "mean"),
        nb_releves=("consommation_kwh_clean", "count")
    )
)

consommation_par_saison

,saison,consommation_totale_kwh,consommation_moyenne_kwh,temp_moyenne_c,degres_jour_chauffage,nb_releves
0,automne,2940630.240,23.081870,13.000896,4.461240,127400
1,ete,2608231.450,20.250244,20.460213,0.131999,128800
2,hiver,3580219.485,28.257454,4.414440,12.585560,126700
3,printemps,3057774.860,23.740488,11.965416,5.339634,128800


In [10]:
# Météo et consommation journalière

meteo_consommation_journaliere = (
    conso
    .groupby("date", as_index=False)
    .agg(
        consommation_totale_kwh=("consommation_kwh_clean", "sum"),
        consommation_moyenne_kwh=("consommation_kwh_clean", "mean"),
        temp_moyenne_c=("temp_moyenne_c", "mean"),
        temp_min_c=("temp_min_c", "mean"),
        temp_max_c=("temp_max_c", "mean"),
        degres_jour_chauffage=("degres_jour_chauffage", "mean"),
        nb_releves=("consommation_kwh_clean", "count")
    )
)

meteo_consommation_journaliere.head()

,date,consommation_totale_kwh,consommation_moyenne_kwh,temp_moyenne_c,temp_min_c,temp_max_c,degres_jour_chauffage,nb_releves
0,2024-01-01,21155.675,30.222393,3.889143,0.669786,9.282471,13.110857,700
1,2024-01-02,21209.150,30.298786,4.060386,-0.906257,9.993514,12.939614,700
2,2024-01-03,20949.730,29.928186,5.193214,1.706614,10.241586,11.806786,700
3,2024-01-04,21531.630,30.759471,2.804043,-1.659914,7.487829,14.195957,700
4,2024-01-05,21302.615,30.432307,2.830414,-0.612214,8.862557,14.169586,700


In [11]:
# Incidents par zone

incidents_par_zone = (
    incidents
    .groupby("zone", as_index=False)
    .agg(
        nb_incidents=("id_incident", "count"),
        duree_totale_minutes=("duree_minutes", "sum"),
        duree_moyenne_minutes=("duree_minutes", "mean"),
        nb_pdl_impactes_total=("nb_pdl_impactes", "sum"),
        nb_pdl_impactes_moyen=("nb_pdl_impactes", "mean")
    )
)

incidents_par_zone["duree_moyenne_minutes"] = incidents_par_zone["duree_moyenne_minutes"].round(2)
incidents_par_zone["nb_pdl_impactes_moyen"] = incidents_par_zone["nb_pdl_impactes_moyen"].round(2)

# Incidents par type

incidents_par_type = (
    incidents
    .groupby("type", as_index=False)
    .agg(
        nb_incidents=("id_incident", "count"),
        duree_totale_minutes=("duree_minutes", "sum"),
        nb_pdl_impactes_total=("nb_pdl_impactes", "sum")
    )
)

# Incidents par cause

incidents_par_cause = (
    incidents
    .groupby("cause", as_index=False)
    .agg(
        nb_incidents=("id_incident", "count"),
        duree_totale_minutes=("duree_minutes", "sum"),
        nb_pdl_impactes_total=("nb_pdl_impactes", "sum")
    )
)

incidents_par_zone.head(), incidents_par_type.head(), incidents_par_cause.head()

(             zone  nb_incidents  duree_totale_minutes  duree_moyenne_minutes  \
 0    Bourg-Ancien            69                  6433                  93.23   
 1    Centre-Ville            45                  4699                 104.42   
 2   Coteaux-Ouest            50                  5208                 104.16   
 3  Parc-Tertiaire            51                  4844                  94.98   
 4     Plateau-Est            43                  5145                 119.65   
 
    nb_pdl_impactes_total  nb_pdl_impactes_moyen  
 0                  31680                 459.13  
 1                  23011                 511.36  
 2                  23772                 475.44  
 3                  24896                 488.16  
 4                  22882                 532.14  ,
                      type  nb_incidents  duree_totale_minutes  \
 0          baisse_tension            81                  8064   
 1                 coupure            76                  7727   
 2  mai

In [12]:
# Réclamations mensuelles

reclamations_mensuelles = (
    reclamations
    .groupby(["annee", "mois"], as_index=False)
    .agg(
        nb_reclamations=("id_reclamation", "count"),
        satisfaction_moyenne=("satisfaction", "mean"),
        nb_clients_concernes=("id_client", "nunique")
    )
)

reclamations_mensuelles["satisfaction_moyenne"] = reclamations_mensuelles["satisfaction_moyenne"].round(2)

reclamations_mensuelles["periode"] = pd.to_datetime(
    reclamations_mensuelles["annee"].astype(str) + "-"
    + reclamations_mensuelles["mois"].astype(str) + "-01"
)

# Réclamations par canal

reclamations_par_canal = (
    reclamations
    .groupby("canal", as_index=False)
    .agg(
        nb_reclamations=("id_reclamation", "count"),
        satisfaction_moyenne=("satisfaction", "mean"),
        longueur_moyenne_texte=("longueur_texte", "mean")
    )
)

reclamations_par_canal["satisfaction_moyenne"] = reclamations_par_canal["satisfaction_moyenne"].round(2)
reclamations_par_canal["longueur_moyenne_texte"] = reclamations_par_canal["longueur_moyenne_texte"].round(2)

reclamations_mensuelles.head(), reclamations_par_canal.head()

(   annee  mois  nb_reclamations  satisfaction_moyenne  nb_clients_concernes  \
 0   2024     1              106                  2.44                   101   
 1   2024     2               88                  2.44                    83   
 2   2024     3               97                  2.64                    93   
 3   2024     4              109                  2.65                   104   
 4   2024     5              104                  2.64                    99   
 
      periode  
 0 2024-01-01  
 1 2024-02-01  
 2 2024-03-01  
 3 2024-04-01  
 4 2024-05-01  ,
            canal  nb_reclamations  satisfaction_moyenne  \
 0       courrier              770                  2.41   
 1          email              742                  2.48   
 2  espace_client              737                  2.42   
 3      telephone              751                  2.48   
 
    longueur_moyenne_texte  
 0                  109.98  
 1                  108.64  
 2                  108.07  
 3 

In [13]:
# Profil horaire moyen

profil_horaire_moyen = (
    releves_horaires
    .groupby("heure", as_index=False)
    .agg(
        consommation_moyenne_kwh=("consommation_kwh", "mean"),
        consommation_mediane_kwh=("consommation_kwh", "median"),
        consommation_totale_kwh=("consommation_kwh", "sum"),
        nb_releves=("consommation_kwh", "count")
    )
)

# Profil horaire par zone

profil_horaire_zone = (
    releves_horaires
    .groupby(["heure", "zone"], as_index=False)
    .agg(
        consommation_moyenne_kwh=("consommation_kwh", "mean"),
        consommation_totale_kwh=("consommation_kwh", "sum"),
        nb_pdl=("id_pdl", "nunique")
    )
)

profil_horaire_moyen.head(), profil_horaire_zone.head()

(   heure  consommation_moyenne_kwh  consommation_mediane_kwh  \
 0      0                  0.794011                      0.21   
 1      1                  0.591333                      0.16   
 2      2                  0.599589                      0.17   
 3      3                  0.606933                      0.17   
 4      4                  0.600844                      0.16   
 
    consommation_totale_kwh  nb_releves  
 0                   714.61         900  
 1                   532.20         900  
 2                   539.63         900  
 3                   546.24         900  
 4                   540.76         900  ,
    heure            zone  consommation_moyenne_kwh  consommation_totale_kwh  \
 0      0    Bourg-Ancien                  0.479667                    14.39   
 1      0    Centre-Ville                  0.143444                    12.91   
 2      0   Coteaux-Ouest                  0.211667                    31.75   
 3      0  Parc-Tertiaire          

In [14]:
# Export des tables agrégées pour Power BI

exports = {
    "powerbi_kpi_global.csv": kpi_global,
    "powerbi_consommation_mensuelle.csv": consommation_mensuelle,
    "powerbi_consommation_par_zone.csv": consommation_par_zone,
    "powerbi_consommation_par_type_client.csv": consommation_par_type_client,
    "powerbi_consommation_par_segment.csv": consommation_par_segment,
    "powerbi_consommation_par_chauffage.csv": consommation_par_chauffage,
    "powerbi_consommation_par_saison.csv": consommation_par_saison,
    "powerbi_meteo_consommation_journaliere.csv": meteo_consommation_journaliere,
    "powerbi_incidents_par_zone.csv": incidents_par_zone,
    "powerbi_incidents_par_type.csv": incidents_par_type,
    "powerbi_incidents_par_cause.csv": incidents_par_cause,
    "powerbi_reclamations_mensuelles.csv": reclamations_mensuelles,
    "powerbi_reclamations_par_canal.csv": reclamations_par_canal,
    "powerbi_profil_horaire_moyen.csv": profil_horaire_moyen,
    "powerbi_profil_horaire_zone.csv": profil_horaire_zone,
    "powerbi_rapport_nettoyage.csv": rapport_nettoyage,
}

for filename, df in exports.items():
    df.to_csv(DATA_CLEANED_DIR / filename, index=False, encoding="utf-8")

print(f"{len(exports)} fichiers exportés dans {DATA_CLEANED_DIR}")

16 fichiers exportés dans c:\Users\Master\Desktop\Mohamed\1CPDA\Examen_S2\ExaS2\neovolt-grid-plus\volet-b-data-analyst\data_cleaned


## Conclusion de la préparation Power BI

Les fichiers agrégés nécessaires au dashboard Power BI ont été générés.

Ils permettent de construire plusieurs vues :
- une vue Direction avec les KPI globaux ;
- une vue Exploitation réseau avec la consommation, les zones, les incidents et les profils horaires ;
- une vue Finance avec les consommations par segment, zone et type client ;
- une vue Relation client avec les réclamations et la satisfaction ;
- une vue Qualité des données avec les indicateurs issus du nettoyage.

Le fichier complet `consommation_preparee.csv` reste local car il est volumineux. Les exports Power BI sont plus légers et adaptés au partage dans GitHub.